# Bag-of-Words (BoW)

## Introduction

This notebook demonstrates the **Bag-of-Words (BoW)** model, a fundamental technique in Natural Language Processing (NLP) for representing text data numerically.

We will:
1.  Explain the concept of Bag-of-Words.
2.  Load a curated dataset (SMS Spam Collection) that requires minimal cleaning.
3.  Apply the BoW technique using Scikit-learn's `CountVectorizer`.
4.  Train a simple classification model (Multinomial Naive Bayes) on the BoW features.
5.  Evaluate the model and make predictions on new data.

## What is Bag-of-Words?

The Bag-of-Words model is a way of representing text data when modeling text with machine learning algorithms. It simplifies text by focusing *only* on the **occurrence (frequency) of words** within a document, completely **disregarding grammar and word order**.

**Core Idea:**

Imagine you have a collection of documents (e.g., sentences, emails, reviews). The BoW model treats each document as a "bag" containing its words.

1.  **Tokenization:** First, the text in each document is broken down into individual words or terms (tokens).
2.  **Vocabulary Building:** A vocabulary of all unique words across *all* documents in the training set is created.
3.  **Vectorization:** Each document is then converted into a numerical vector. The length of this vector is equal to the size of the vocabulary. Each position (index) in the vector corresponds to a unique word in the vocabulary. The value at that position represents how many times that specific word appears in the document.

**Example:**

Consider these two sentences:

*   Sentence 1: "The cat sat on the mat."
*   Sentence 2: "The dog chased the cat."

1.  **Tokens:**
    *   Sentence 1: `['the', 'cat', 'sat', 'on', 'the', 'mat']`
    *   Sentence 2: `['the', 'dog', 'chased', 'the', 'cat']`
2.  **Vocabulary:** `['the', 'cat', 'sat', 'on', 'mat', 'dog', 'chased']` (size 7)
3.  **Vectors (Counts):**
    *   Sentence 1: `[2, 1, 1, 1, 1, 0, 0]` (2 'the', 1 'cat', 1 'sat', 1 'on', 1 'mat', 0 'dog', 0 'chased')
    *   Sentence 2: `[2, 1, 0, 0, 0, 1, 1]` (2 'the', 1 'cat', 0 'sat', 0 'on', 0 'mat', 1 'dog', 1 'chased')

**Key Takeaway:** BoW converts variable-length text documents into fixed-size numerical vectors, making them suitable for machine learning algorithms. However, it loses information about word order and context.

## Setup

Import necessary libraries.

In [9]:
#!pip install pandas
# !pip install scikit-learn

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

## Load Data

We'll use the SMS Spam Collection dataset, which is well-suited for this task as it's relatively clean. It contains SMS messages labeled as either 'ham' (legitimate) or 'spam'.

In [11]:
# Load the dataset from a URL
# Dataset source: https://archive.ics.uci.edu/ml/datasets/SMS+Spam+Collection
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

# Display the first few rows and info
print("Dataset Head:")
print(df.head())
print("\nDataset Info:")
df.info()

Dataset Head:
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Dataset Info:
<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572 non-null   str  
dtypes: str(2)
memory usage: 542.9 KB


## Prepare Data

1.  Separate features (the SMS message text) and the target variable (the label).
2.  Map the categorical labels ('ham', 'spam') to numerical values (0, 1), as models work with numbers.
3.  Split the data into training and testing sets to evaluate the model's performance on unseen data.

In [12]:
# Map labels to numerical values
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

# Define features (X) and target (y)
X = df['message']
y = df['label_num']

X.head()

0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: message, dtype: str

In [13]:
y.head()

0    0
1    0
2    1
3    0
4    0
Name: label_num, dtype: int64

In [14]:
# Split data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]} messages")
print(f"Test set size: {X_test.shape[0]} messages")

print("-----------------------")

print("X Head:")
print(X.head())

print("-----------------------")

print("y Head:")
print(y.head())

Training set size: 4457 messages
Test set size: 1115 messages
-----------------------
X Head:
0    Go until jurong point, crazy.. Available only ...
1                        Ok lar... Joking wif u oni...
2    Free entry in 2 a wkly comp to win FA Cup fina...
3    U dun say so early hor... U c already then say...
4    Nah I don't think he goes to usf, he lives aro...
Name: message, dtype: str
-----------------------
y Head:
0    0
1    0
2    1
3    0
4    0
Name: label_num, dtype: int64


## Applying Bag-of-Words (Vectorization)

Now, we'll convert the text messages into numerical vectors using the BoW approach. We use Scikit-learn's `CountVectorizer`.

**Key Steps:**

1.  **Initialize `CountVectorizer`**: We create an instance of the vectorizer.
2.  **Fit**: We `fit` the vectorizer **only on the training data (`X_train`)**. This step builds the vocabulary based on the words present in the training messages.
3.  **Transform**: We then `transform` both the training data (`X_train`) and the test data (`X_test`) using the *fitted* vectorizer. This converts each message into a sparse matrix where rows represent messages and columns represent words from the vocabulary, with cell values indicating word counts.

**Important:** We fit *only* on the training data to prevent data leakage from the test set into the vocabulary building process. The test set should remain unseen until evaluation.

In [15]:
# 1. Initialize CountVectorizer
# We can add parameters like stop_words='english' to remove common English words,
# or max_features to limit the vocabulary size, but we'll keep it simple for now.
vectorizer = CountVectorizer()

# 2. Fit the vectorizer on the training data to build the vocabulary
vectorizer.fit(X_train)

# Print vocabulary size
print(f"Vocabulary Size: {len(vectorizer.vocabulary_)}")

# Optionally, view a sample of the vocabulary
# print("Sample Vocabulary:", list(vectorizer.vocabulary_.items())[:10])

# 3. Transform the training and test data into BoW vectors (sparse matrix)
X_train_bow = vectorizer.transform(X_train)
X_test_bow = vectorizer.transform(X_test)

# Print the shape of the resulting BoW matrices
# Shape: (number of messages, vocabulary size)
print(f"Shape of Training BoW matrix: {X_train_bow.shape}")
print(f"Shape of Test BoW matrix: {X_test_bow.shape}")

# Show the BoW representation for the first training message (as a dense array for readability)
print("\nBoW vector for the first training message (sparse):")
print(X_train_bow[0])
print("\nBoW vector for the first training message (dense, showing first 50 features):")
print(X_train_bow[0].toarray()[0, :50])

Vocabulary Size: 7668
Shape of Training BoW matrix: (4457, 7668)
Shape of Test BoW matrix: (1115, 7668)

BoW vector for the first training message (sparse):
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 5 stored elements and shape (1, 7668)>
  Coords	Values
  (0, 1861)	1
  (0, 3290)	1
  (0, 3373)	1
  (0, 7439)	1
  (0, 7629)	1

BoW vector for the first training message (dense, showing first 50 features):
[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]


The output `(0, word_index) count` from the sparse matrix means: in the first document (index 0), the word corresponding to `word_index` in the vocabulary appeared `count` times.

## Training a Simple Model

Now that our text data is represented numerically using Bag-of-Words, we can train a classification model.

We'll use **Multinomial Naive Bayes (MNB)**, which is often a good baseline model for text classification tasks, especially with count-based features like BoW.

The model learns the probability of words appearing in 'ham' vs 'spam' messages based on the counts provided by the BoW vectors.

In [16]:
# Initialize the Multinomial Naive Bayes classifier
model = MultinomialNB()

# Train the model using the BoW representation of the training data
model.fit(X_train_bow, y_train)

print("Model training complete.")

Model training complete.


## Making Predictions

We use the trained model to predict the labels ('ham' or 'spam') for the messages in the **test set** (`X_test_bow`).

In [17]:
# Predict labels for the test set
y_pred = model.predict(X_test_bow)

print("Predictions made on the test set.")
# print("Sample predictions:", y_pred[:10])
# print("Actual labels:", y_test[:10].values)

Predictions made on the test set.


## Evaluating the Model

Let's evaluate how well our model performed using standard classification metrics.

In [18]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

# Display the confusion matrix
print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
# Rows: Actual (0: Ham, 1: Spam)
# Columns: Predicted (0: Ham, 1: Spam)

# Display the classification report (precision, recall, f1-score)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

Accuracy: 0.9874

Confusion Matrix:
[[964   2]
 [ 12 137]]

Classification Report:
              precision    recall  f1-score   support

         Ham       0.99      1.00      0.99       966
        Spam       0.99      0.92      0.95       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115



**Interpretation:**

*   **Accuracy:** Overall percentage of correct predictions.
*   **Confusion Matrix:** Shows correct and incorrect predictions for each class.
    *   Top-left: True Negatives (Correctly predicted 'Ham')
    *   Bottom-right: True Positives (Correctly predicted 'Spam')
    *   Top-right: False Positives (Predicted 'Spam' but was 'Ham' - Type I error)
    *   Bottom-left: False Negatives (Predicted 'Ham' but was 'Spam' - Type II error)
*   **Classification Report:**
    *   **Precision:** Of all messages predicted as Spam, how many actually were Spam? (TP / (TP + FP))
    *   **Recall:** Of all actual Spam messages, how many did the model correctly identify? (TP / (TP + FN))
    *   **F1-Score:** Harmonic mean of Precision and Recall, useful for imbalanced datasets.

## Example Prediction on New Data

Let's see how the model classifies a couple of new messages. Remember, we need to transform these new messages using the **same fitted `vectorizer`** that we used for the training data.

In [21]:
# New messages to classify
new_messages = [
    "Hello, how are you today? Let's meet for lunch.",
    "Congratulations! You've won a $1000 gift card. Click here to claim.",
    "Hello, you've won a free car"
]

# Transform the new messages using the *fitted* vectorizer
new_messages_bow = vectorizer.transform(new_messages)

# Predict the labels for the new messages
new_predictions = model.predict(new_messages_bow)

# Map numerical predictions back to labels
label_map = {0: 'Ham', 1: 'Spam'}
predicted_labels = [label_map[pred] for pred in new_predictions]

# Print the results
for msg, label in zip(new_messages, predicted_labels):
    print(f'Message: "{msg}" -> Predicted Label: {label}')

Message: "Hello, how are you today? Let's meet for lunch." -> Predicted Label: Ham
Message: "Congratulations! You've won a $1000 gift card. Click here to claim." -> Predicted Label: Spam
Message: "Hello, you've won a free car" -> Predicted Label: Ham


## Conclusion

This notebook demonstrated the Bag-of-Words model for text representation.

*   We converted text messages into numerical vectors based on word counts using `CountVectorizer`.
*   These vectors allowed us to train a `MultinomialNB` classifier for spam detection.
*   The model achieved good performance on the test set, showing the effectiveness of BoW for this task.

**Limitations of BoW:**

*   It ignores word order and context (e.g., "not good" is treated similarly to "good").
*   It doesn't capture semantic meaning (e.g., 'car' and 'automobile' are treated as different words).
*   The resulting vectors can be very high-dimensional (large vocabulary) and sparse.

Despite these limitations, BoW is a simple, efficient, and often effective baseline for many NLP tasks.